**Set environment**

In [1]:
import numpy  as np
import pandas as pd
import itertools as it
from functools import partial
import os, sys, re
import csv

In [2]:
%run ../run_config_project.py
show_env()

BASE DIRECTORY (FD_BASE): /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO): /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK): /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA): /hpc/group/igvf/kk319/data


You are working with      IGVF BlueSTARR
PATH OF PROJECT (FD_PRJ): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references



In [3]:
FP_GEN = "/hpc/group/igvf/kk319/data/genome/hg38/hg38.fa"

import pysam
fasta  = pysam.FastaFile(FP_GEN)

#from pyfaidx import Fasta
#fasta  = Fasta(FP_GEN)

## Import data

In [4]:
### set file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")
txt_fname = "variant_closed_gof_bluestarr.tsv"
txt_fpath = os.path.join(txt_fdiry, txt_fname)

### read table
dat = pd.read_csv(txt_fpath, sep = "\t")

### assign and show
dat_variant_import = dat
print(dat.shape)
dat.head()

(1884977, 10)


,Chrom,ChromStart,ChromEnd,Region,Variant_ID,Pos0,Ref,Obs,Unobs,Delta
0,chr4,74487576,74488141,chr4:74487576-74488141,chr4:74487586:A:G:C,74487586,A,G,C,0.964984
1,chr9,135685425,135685942,chr9:135685425-135685942,chr9:135685432:G:G:C,135685432,G,G,C,0.945423
2,chr1,173251481,173251541,chr1:173251481-173251541,chr1:173251490:G:G:C,173251490,G,G,C,0.910127
3,chr1,147523084,147523503,chr1:147523084-147523503,chr1:147523391:G:G:C,147523391,G,G,C,0.908800
4,chr18,54580694,54581420,chr18:54580694-54581420,chr18:54580749:A:G:T,54580749,A,G,T,0.862296


## Sanity check: reference allele

Check if the reference allele column is correct

**Check the first row**

In [5]:
txt_region = "chr4:74487576-74488141"

txt_chrom_name, txt_chrom_slice = txt_region.split(":")
txt_chrom_name  = str(txt_chrom_name)
txt_chrom_pos   = 74487586

print(f"Chrom: {txt_chrom_name}; Start-End: {txt_chrom_slice}; Pos: {txt_chrom_pos}")

Chrom: chr4; Start-End: 74487576-74488141; Pos: 74487586


In [6]:
### # 0-based slicing
print(fasta.fetch(txt_chrom_name, txt_chrom_pos, txt_chrom_pos+1))
#print(fasta[txt_chrom_name][txt_chrom_pos:txt_chrom_pos+1].seq)

A


**Check more positions**

In [7]:
%%time
### init
dat = dat_variant_import
dat = dat.sample(1000, random_state=1)

### fetch ref allele and check mismatches
lst_mismatches = []

for txt_chrom_name, num_chrom_pos, txt_allele_ref_table in zip(
    dat["Chrom"],
    dat["Pos0"], 
    dat["Ref"]
):
    ### query the reference allele
    ### note: pos is 0-based, end-exclusive
    txt_allele_ref_fetch = fasta.fetch(txt_chrom_name, num_chrom_pos, num_chrom_pos+1) 

    ### sequence has uppercase and lowercase bases
    ### - uppercase: high-confidence sequence
    ### - lowercase: soft-masked sequence
    ### convert all to uppercase for simplicity
    txt_allele_ref_fetch = txt_allele_ref_fetch.upper()
    
    ### check match/mismatch
    if txt_allele_ref_fetch != txt_allele_ref_table:
        tmp = (txt_chrom_name, num_chrom_pos, txt_allele_ref_table, txt_allele_ref_fetch)
        lst_mismatches.append(tmp)

print(f"Checked {len(dat)} variants")
print(f"Found {len(lst_mismatches)} mismatches")
if lst_mismatches:
    print("Example mismatches:", lst_mismatches[:5])

Checked 1000 variants
Found 0 mismatches
CPU times: user 172 ms, sys: 95 ms, total: 267 ms
Wall time: 12.8 s


## Helper function

In [8]:
def get_interval_refseq(
    txt_chrom_name, num_chrom_pos0, txt_allele_ref,
    num_interval_flank_left=35, 
    num_interval_flank_right=35
):
    ### fetch genomic interval centered at variant position
    txt_seq_ref = fasta.fetch(
        txt_chrom_name,
        num_chrom_pos0 - num_interval_flank_left,
        num_chrom_pos0 + num_interval_flank_right + 1
    ).upper()

    ### sanity check reference allele
    if txt_seq_ref[num_interval_flank_left].upper() != txt_allele_ref.upper():
        raise ValueError(
            f"Reference mismatch at {txt_chrom_name}:{txt_chrom_pos0} "
            f"(expected {txt_allele_ref}, got {txt_seq_ref[num_interval_flank_left]})"
        )

    ### return sequence
    return txt_seq_ref

## Get sequence by specifying flanking size as L35bp R70bp

In [9]:
### spanning x bp at the center of each variant
### here the flanking region is set as 35 bp at both side
NUM_INTERVAL_FLANK_LEFT  = 35 
NUM_INTERVAL_FLANK_RIGHT = 70 

**Test run**

In [10]:
dat.iloc[0,:]

Chrom                             chr4
ChromStart                   183063682
ChromEnd                     183063779
Region        chr4:183063682-183063779
Variant_ID        chr4:183063712:T:C:A
Pos0                         183063712
Ref                                  T
Obs                                  C
Unobs                                A
Delta                         0.006892
Name: 771970, dtype: object

In [11]:
%%time

### init: get random 100 variants
dat = dat_variant_import.copy()
dat = dat.sample(100, random_state=123)
lst_txt_seq = []

### add sequences
fun = partial(
    get_interval_refseq, 
    num_interval_flank_left  = NUM_INTERVAL_FLANK_LEFT,
    num_interval_flank_right = NUM_INTERVAL_FLANK_RIGHT
)

### loop through each variant and get refseq
for txt_chrom_name, num_chrom_pos0, txt_allele_ref in zip(
    dat["Chrom"],
    dat["Pos0"],
    dat["Ref"]
):
    txt_seq = fun(
        txt_chrom_name,
        int(num_chrom_pos0),
        txt_allele_ref
    )
    lst_txt_seq.append(txt_seq)

### assign and show
dat["Seq_Ref"] = lst_txt_seq
dat.head()

CPU times: user 229 ms, sys: 22.4 ms, total: 251 ms
Wall time: 1.04 s


,Chrom,ChromStart,ChromEnd,Region,Variant_ID,Pos0,Ref,Obs,Unobs,Delta,Seq_Ref
1467403,chr2,181868375,181869091,chr2:181868375-181869091,chr2:181869026:G:A:T,181869026,G,A,T,0.001980,CTTGATCCTCATAATCCTCACAATAACTCTAGAGAGTGAGAAGTGT...
176754,chr1,159747007,159747231,chr1:159747007-159747231,chr1:159747115:T:T:C,159747115,T,T,C,0.018575,CGTGCATTTTTGTGACCCCAATCATTTTTGAAAACTATCTCAGAGC...
227022,chr11,55372676,55374644,chr11:55372676-55374644,chr11:55374072:T:A:C,55374072,T,A,C,0.016494,AGCCACCTCCTAAAATATATTTGTATACATACTATTTCTATTTTAT...
471983,chr7,73751160,73751171,chr7:73751160-73751171,chr7:73751165:G:G:T,73751165,G,G,T,0.010665,TAAGTTTCAAGCTTTAAGACCAAAAGGGTTGACCCGCACCATGGCT...
1118365,chr8,72987671,72988177,chr8:72987671-72988177,chr8:72987743:G:G:T,72987743,G,G,T,0.004071,TTAATTAAAGATTACAAACAAAGCTGAAAACCTAGGAGAAGATAGG...


**Full run; export to fasta file**

In [12]:
def fun_wrap_fasta(seq, width=60):
    """
    Wrap a sequence string into fixed-width lines for FASTA output.
    """
    return "\n".join(
        seq[i:i + width]
        for i in range(0, len(seq), width)
    )

In [13]:
### set file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")
txt_fname = "variant_closed_gof_bluestarr.flankL35R70.ref.fa"
txt_fpath = os.path.join(txt_fdiry, txt_fname)

### init: function
fun = partial(
    get_interval_refseq, 
    num_interval_flank_left  = NUM_INTERVAL_FLANK_LEFT,
    num_interval_flank_right = NUM_INTERVAL_FLANK_RIGHT
)

### loop through each variant and get refseq
with open(txt_fpath, "w") as fout:
    for idx, (txt_chrom_name, num_chrom_pos0, txt_allele_ref, txt_variant_idx) in enumerate(
        zip(
            dat_variant_import["Chrom"],
            dat_variant_import["Pos0"],
            dat_variant_import["Ref"],
            dat_variant_import["Variant_ID"],
        ),
        start=1
    ):
        ### verbose progress
        if idx % 100_000 == 0:
            print(f"Written {idx:,} sequences...")

        ### get refseq
        txt_seq = fun(
            txt_chrom_name,
            int(num_chrom_pos0),
            txt_allele_ref
        )
        
        ### output refseq
        fout.write(f">{txt_variant_idx}\n")
        fout.write(fun_wrap_fasta(txt_seq, width=60) + "\n")

print("Wrote:", txt_fpath)

Written 100,000 sequences...
Written 200,000 sequences...
Written 300,000 sequences...
Written 400,000 sequences...
Written 500,000 sequences...
Written 600,000 sequences...
Written 700,000 sequences...
Written 800,000 sequences...
Written 900,000 sequences...
Written 1,000,000 sequences...
Written 1,100,000 sequences...
Written 1,200,000 sequences...
Written 1,300,000 sequences...
Written 1,400,000 sequences...
Written 1,500,000 sequences...
Written 1,600,000 sequences...
Written 1,700,000 sequences...
Written 1,800,000 sequences...
Wrote: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/variant_closed_gof_bluestarr.flankL35R70.ref.fa
